In [5]:
"""
Combined model (post + HA features) with Content_Sharing as reference class.
Outputs:
  - full_results_with_constants.csv   : all coefficients including intercepts
  - significant_results.csv           : rows where p < 0.05
  - summary.txt                       : model fit statistics
  - forest_plot_significant.png       : coefficient forest plot (sig. results only)
"""

import os
import warnings
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

# ─── Paths ────────────────────────────────────────────────────────────────────
DATA_PATH = r"C:\Users\20245179\OneDrive - TU Eindhoven\Research Paper\Data\KMeans_filtered_with_HA_feature_levels_top50.csv"
OUT_DIR   = r"C:\Users\20245179\Desktop\EngD_WP1_AnSoME_analyze_social_media\MLT\combined_content_sharing_ref"

# ─── Constants ────────────────────────────────────────────────────────────────
DISCOURSE_MAP = {
    0: "On-topic_Feedbacks",
    1: "On-topic_Criticism",
    2: "On-topic_Praise",
    3: "Off-topic_Complaints",
    4: "Content_Sharing",        # <-- reference category
    5: "Information_Seeking",
}
CLUSTERS     = list(DISCOURSE_MAP.keys())   # [0, 1, 2, 3, 4, 5]
REF_CLUSTER  = 4                            # Content_Sharing

# Post-length bins
LENGTH_BINS   = [0, 50, 150, np.inf]
LENGTH_LABELS = ["Very_Short", "Short", "Medium_Long"]
LENGTH_REF    = "Very_Short"               # dropped in dummies → reference

# Unique-words ratio: reference = "Low"
UW_REF = "Low"

# HA features included in the model
HA_FEATURES = [
    # "Operation expense/unit",
    # "Maintainance cost/unit",
    "Total Housing",
    "Affordability(Rent/month)",
    "Tenants Left",
]

# Inverted vars: low raw value = better outcome → label flipped so "High" = better
INVERTED_VARS = {
    "Affordability(Rent/month)": "Affordability_Level",
    "Tenants Left"             : "Tenant_Satisfaction_Level",
}

# Reference level for all HA dummies = "Low"
HA_REF_LEVEL = "Low"

# Significance thresholds
SIG_THRESHOLD = 0.05
SIG_LABELS    = {0.001: "***", 0.01: "**", 0.05: "*", 0.10: "."}

# ─── Load & prepare data ──────────────────────────────────────────────────────
print("Loading data …")
ha_df    = pd.read_csv(DATA_PATH)
df_model = ha_df.copy()
df_model["Post"] = df_model["Post"].astype(str)

# ── Post features
df_model["post_length"]        = df_model["Post"].apply(lambda x: len(x.split()))
df_model["length_category"]    = pd.cut(
    df_model["post_length"], bins=LENGTH_BINS, labels=LENGTH_LABELS,
    right=True, include_lowest=True
)
df_model["unique_words"]       = df_model["Post"].apply(lambda x: len(set(x.split())))
df_model["unique_words_ratio"] = (
    df_model["unique_words"] / df_model["post_length"] * 100
).round(2).fillna(0)

q33_uw = df_model["unique_words_ratio"].quantile(0.33)
df_model["unique_words_cat"] = df_model["unique_words_ratio"].apply(
    lambda x: "Low" if x < q33_uw else "Medium_High"
)
df_model["has_questions"] = df_model["Post"].str.contains(r"\?", regex=True).astype(int)
df_model["has_urls"]      = df_model["Post"].str.contains(r"http\S+", regex=True).astype(int)

# ── HA features
HA_LEVEL_FEATS = []
for feat in HA_FEATURES:
    q33_ha = df_model[feat].quantile(0.33)
    q67_ha = df_model[feat].quantile(0.67)
    if feat in INVERTED_VARS:
        level_col = INVERTED_VARS[feat]
        df_model[level_col] = pd.cut(
            df_model[feat], bins=[-np.inf, q67_ha, np.inf],
            labels=["Medium_High", "Low"]   # intentionally reversed: low raw → High level
        )
    else:
        level_col = f"{feat}_Level"
        df_model[level_col] = pd.cut(
            df_model[feat], bins=[-np.inf, q33_ha, np.inf],
            labels=["Low", "Medium_High"]
        )
    HA_LEVEL_FEATS.append(level_col)

# Deduplicate to unique posts
df_posts = df_model.drop_duplicates(subset="Post").copy()
print(f"Unique posts: {len(df_posts)}")


# ─── Build feature matrix ─────────────────────────────────────────────────────
def build_X(df):
    parts = []

    # unique_words_cat  (ref = Low → drop uw_Low)
    uw = pd.get_dummies(df["unique_words_cat"], prefix="uw").astype(float)
    if f"uw_{UW_REF}" in uw.columns:
        uw = uw.drop(columns=[f"uw_{UW_REF}"])
    parts.append(uw)

    # length_category  (ref = Very_Short → drop length_Very_Short)
    lc = pd.get_dummies(df["length_category"], prefix="length").astype(float)
    ref_col = [c for c in lc.columns if LENGTH_REF in c]
    if ref_col:
        lc = lc.drop(columns=ref_col)
    parts.append(lc)

    # binary post features
    parts.append(df[["has_questions", "has_urls"]].astype(float))

    # HA level features  (ref = Low for each)
    for feat in HA_LEVEL_FEATS:
        safe = (feat.replace("(", "").replace(")", "")
                    .replace("/", "_").replace(" ", "_"))
        d = pd.get_dummies(df[feat], prefix=safe).astype(float)
        ref_col = [c for c in d.columns if c.endswith(f"_{HA_REF_LEVEL}")]
        if ref_col:
            d = d.drop(columns=ref_col)
        parts.append(d)

    return pd.concat(parts, axis=1)


# ─── Fit model & extract results (including constants) ────────────────────────
def fit_model(df, ref_cluster):
    other_clusters = [c for c in CLUSTERS if c != ref_cluster]
    cluster_order  = [ref_cluster] + other_clusters
    class_names    = [DISCOURSE_MAP[c] for c in cluster_order]

    y = df["Cluster_KMeans"].map({c: i for i, c in enumerate(cluster_order)}).values
    X = build_X(df)
    X_const = sm.add_constant(X)

    model   = sm.MNLogit(y, X_const)
    results = model.fit(disp=False, maxiter=500)

    feature_names = list(X.columns)
    all_features  = ["const"] + feature_names   # include intercept

    records = []
    for cls_idx, cls_name in enumerate(class_names[1:]):
        cls_num = cluster_order[cls_idx + 1]
        # params shape: (n_features+1, n_classes-1)  → rows include const at index 0
        coeffs = results.params.values[:, cls_idx]
        pvals  = results.pvalues.values[:, cls_idx]
        se     = results.bse.values[:, cls_idx]

        for feat, coef, pv, std in zip(all_features, coeffs, pvals, se):
            # Determine significance marker
            sig = ""
            for thresh, marker in sorted(SIG_LABELS.items()):
                if pv < thresh:
                    sig = marker
                    break

            records.append({
                "Reference_Cluster" : ref_cluster,
                "Reference_Label"   : DISCOURSE_MAP[ref_cluster],
                "Discourse_Type"    : cls_name,
                "Cluster"           : cls_num,
                "Feature"           : feat,
                "Coef"              : round(coef, 4),
                "SE"                : round(std, 4),
                "p_value"           : round(pv, 4),
                "Sig"               : sig,
                "Significant"       : pv < SIG_THRESHOLD,
                "Is_Constant"       : feat == "const",
            })

    return pd.DataFrame(records), results, class_names


# ─── Save CSVs and summary ────────────────────────────────────────────────────
def save_outputs(results_df, fit_results, class_names, ref_cluster, folder):
    os.makedirs(folder, exist_ok=True)

    # Full table (including constants)
    results_df.to_csv(
        os.path.join(folder, "full_results_with_constants.csv"), index=False
    )

    # Significant rows only (excluding constants for clarity)
    sig_df = (
        results_df[results_df["Significant"] & ~results_df["Is_Constant"]]
        .sort_values(["Discourse_Type", "p_value"])
    )
    sig_df.to_csv(os.path.join(folder, "significant_results.csv"), index=False)

    # Text summary
    ref_label = DISCOURSE_MAP[ref_cluster]
    with open(os.path.join(folder, "summary.txt"), "w") as f:
        f.write(f"Reference class : {ref_label} (Cluster {ref_cluster})\n")
        f.write(f"N posts         : {int(fit_results.nobs)}\n")
        f.write(f"Pseudo R2       : {fit_results.prsquared:.4f}\n")
        f.write(f"AIC             : {fit_results.aic:.2f}\n")
        f.write(f"BIC             : {fit_results.bic:.2f}\n")
        f.write(f"Log-Likelihood  : {fit_results.llf:.2f}\n")
        f.write(f"Converged       : {fit_results.mle_retvals.get('converged', 'N/A')}\n\n")

        predictor_rows = results_df[~results_df["Is_Constant"]]
        n_sig = predictor_rows["Significant"].sum()
        n_tot = len(predictor_rows)
        f.write(f"Significant     : {n_sig}/{n_tot} ({n_sig/n_tot*100:.1f}%)\n\n")

        # Constants section
        f.write("=" * 72 + "\n")
        f.write("CONSTANTS (Intercepts)\n")
        f.write("=" * 72 + "\n")
        hdr = f"{'Discourse Type':<30} {'Const':>10} {'SE':>8} {'p':>8}  Sig\n"
        f.write(hdr)
        f.write("-" * 70 + "\n")
        for _, row in results_df[results_df["Is_Constant"]].iterrows():
            f.write(
                f"{row['Discourse_Type']:<30} {row['Coef']:>10.4f} "
                f"{row['SE']:>8.4f} {row['p_value']:>8.4f}  {row['Sig']}\n"
            )

        # Significant predictors section
        f.write("\n" + "=" * 72 + "\n")
        f.write("SIGNIFICANT PREDICTORS (p < 0.05, excluding constants)\n")
        f.write("=" * 72 + "\n")
        hdr = f"{'Discourse Type':<30} {'Feature':<35} {'Coef':>8} {'p':>8}  Sig\n"
        f.write(hdr)
        f.write("-" * 88 + "\n")
        for _, row in sig_df.iterrows():
            f.write(
                f"{row['Discourse_Type']:<30} {row['Feature']:<35} "
                f"{row['Coef']:>8.4f} {row['p_value']:>8.4f}  {row['Sig']}\n"
            )

    return sig_df


# ─── Forest plot for significant results ──────────────────────────────────────
def plot_significant(sig_df, ref_label, folder):
    if sig_df.empty:
        print("  No significant results to plot.")
        return

    # ── colour palette per discourse type
    discourse_types = sig_df["Discourse_Type"].unique()
    palette = plt.cm.tab10.colors
    color_map = {dt: palette[i % len(palette)] for i, dt in enumerate(discourse_types)}

    # ── compute 95 % CI
    plot_df = sig_df.copy().reset_index(drop=True)
    plot_df["ci_lo"] = plot_df["Coef"] - 1.96 * plot_df["SE"]
    plot_df["ci_hi"] = plot_df["Coef"] + 1.96 * plot_df["SE"]
    plot_df["label"] = plot_df["Feature"] + "  (" + plot_df["Discourse_Type"] + ")"

    n = len(plot_df)
    fig, ax = plt.subplots(figsize=(10, max(4, n * 0.55 + 1.5)))

    y_pos = range(n - 1, -1, -1)   # top-to-bottom order

    for i, (_, row) in enumerate(plot_df.iterrows()):
        y = list(y_pos)[i]
        color = color_map[row["Discourse_Type"]]

        # CI line
        ax.plot([row["ci_lo"], row["ci_hi"]], [y, y],
                color=color, linewidth=1.5, zorder=2)
        # Point estimate
        ax.scatter(row["Coef"], y, color=color, s=70, zorder=3)
        # Sig marker
        ax.text(row["ci_hi"] + 0.03, y, row["Sig"],
                va="center", ha="left", fontsize=10, color=color)

    # Reference line at 0
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--", zorder=1)

    # y-axis labels
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(plot_df["label"].tolist(), fontsize=9)

    # Legend
    patches = [
        mpatches.Patch(color=color_map[dt], label=dt)
        for dt in discourse_types
    ]
    ax.legend(handles=patches, title="Discourse Type",
              bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)

    ax.set_xlabel("Coefficient (log-odds)  ±  95 % CI", fontsize=11)
    ax.set_title(
        f"Significant predictors (p < 0.05)\nReference: {ref_label}",
        fontsize=12, fontweight="bold"
    )
    ax.grid(axis="x", linestyle=":", alpha=0.5)
    plt.tight_layout()

    plot_path = os.path.join(folder, "forest_plot_significant.png")
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Forest plot saved -> {plot_path}")


# ─── Main ─────────────────────────────────────────────────────────────────────
print(f"\nRunning COMBINED model  |  Reference: {DISCOURSE_MAP[REF_CLUSTER]} (Cluster {REF_CLUSTER})")

res_df, fit_res, class_names = fit_model(df_posts, REF_CLUSTER)

sig_df = save_outputs(res_df, fit_res, class_names, REF_CLUSTER, OUT_DIR)

predictor_rows = res_df[~res_df["Is_Constant"]]
n_sig = predictor_rows["Significant"].sum()
n_tot = len(predictor_rows)
print(f"  Converged  : {fit_res.mle_retvals.get('converged', 'N/A')}")
print(f"  Pseudo R²  : {fit_res.prsquared:.4f}")
print(f"  AIC        : {fit_res.aic:.2f}")
print(f"  Significant: {n_sig}/{n_tot} ({n_sig/n_tot*100:.1f}%)")

plot_significant(sig_df, DISCOURSE_MAP[REF_CLUSTER], OUT_DIR)

# ── Print full table to console ──
print("\n" + "=" * 88)
print("FULL RESULTS TABLE (including constants)")
print("=" * 88)
print(res_df.to_string(index=False))

print(f"\nAll outputs saved to: {OUT_DIR}")
print("Done!")


Loading data …
Unique posts: 648

Running COMBINED model  |  Reference: Content_Sharing (Cluster 4)
  Converged  : True
  Pseudo R²  : 0.0358
  AIC        : 2051.90
  Significant: 8/40 (20.0%)
  Forest plot saved -> C:\Users\20245179\Desktop\EngD_WP1_AnSoME_analyze_social_media\MLT\combined_content_sharing_ref\forest_plot_significant.png

FULL RESULTS TABLE (including constants)
 Reference_Cluster Reference_Label       Discourse_Type  Cluster                               Feature    Coef     SE  p_value Sig  Significant  Is_Constant
                 4 Content_Sharing   On-topic_Feedbacks        0                                 const  2.0073 0.9308   0.0310   *         True         True
                 4 Content_Sharing   On-topic_Feedbacks        0                        uw_Medium_High -0.0948 0.5545   0.8643            False        False
                 4 Content_Sharing   On-topic_Feedbacks        0                          length_Short  0.5563 0.4561   0.2226            False    

In [6]:

# ─── Composition Table ────────────────────────────────────────────────────────
# Counts and percentages for each post feature and HA feature level.
# Runs on df_posts (unique posts) and df_model (all rows before dedup).

print("=" * 60)
print("DATASET TOTALS")
print("=" * 60)
total_rows  = len(ha_df)
total_posts = len(df_posts)

# Detect post vs comment column (common names)
post_comment_col = None
for col in ["type", "Type", "post_type", "PostType", "is_comment", "kind"]:
    if col in ha_df.columns:
        post_comment_col = col
        break

if post_comment_col:
    vc = ha_df[post_comment_col].value_counts()
    print(f"Total rows          : {total_rows:,}")
    for label, cnt in vc.items():
        print(f"  {label:<20}: {cnt:,}  ({cnt/total_rows*100:.1f}%)")
else:
    print(f"Total rows in CSV   : {total_rows:,}")
    print(f"Unique posts (dedup): {total_posts:,}")
    print("  (No post/comment type column found — reporting unique posts only)")

print(f"\nRows used in model  : {total_posts:,}  (after dedup on 'Post')")


def feature_table(df, col, label, total=None):
    """Print count & % for each level of a categorical/binary column."""
    if total is None:
        total = len(df)
    vc = df[col].value_counts(dropna=False).sort_index()
    print(f"\n  {label}")
    print(f"  {'Level':<25} {'N':>7}  {'%':>7}")
    print(f"  {'-'*42}")
    for lvl, cnt in vc.items():
        pct = cnt / total * 100
        print(f"  {str(lvl):<25} {cnt:>7,}  {pct:>6.1f}%")


print("\n" + "=" * 60)
print("POST FEATURES  (N = unique posts)")
print("=" * 60)

feature_table(df_posts, "length_category",  "Post Length Category")
feature_table(df_posts, "unique_words_cat",  "Unique-Words Ratio (Low vs Medium_High)")

# has_questions and has_urls are binary → show Yes/No
for col, lbl in [("has_questions", "Has Questions (?)"),
                 ("has_urls",      "Has URLs (http…)")]:
    tmp = df_posts[col].map({1: "Yes (=1)", 0: "No (=0)"})
    feature_table(df_posts.assign(**{col: tmp}), col, lbl)


print("\n" + "=" * 60)
print("HA FEATURES  (N = unique posts)")
print("=" * 60)

for feat_col in HA_LEVEL_FEATS:
    feature_table(df_posts, feat_col, feat_col)


print("\n" + "=" * 60)
print("DISCOURSE TYPES  (N = unique posts)")
print("=" * 60)

disc = df_posts["Cluster_KMeans"].map(DISCOURSE_MAP)
feature_table(df_posts.assign(Discourse_Type=disc), "Discourse_Type", "Discourse Type")


DATASET TOTALS
Total rows in CSV   : 2,882
Unique posts (dedup): 648
  (No post/comment type column found — reporting unique posts only)

Rows used in model  : 648  (after dedup on 'Post')

POST FEATURES  (N = unique posts)

  Post Length Category
  Level                           N        %
  ------------------------------------------
  Very_Short                    213    32.9%
  Short                         389    60.0%
  Medium_Long                    46     7.1%

  Unique-Words Ratio (Low vs Medium_High)
  Level                           N        %
  ------------------------------------------
  Low                           191    29.5%
  Medium_High                   457    70.5%

  Has Questions (?)
  Level                           N        %
  ------------------------------------------
  No (=0)                       358    55.2%
  Yes (=1)                      290    44.8%

  Has URLs (http…)
  Level                           N        %
  ------------------------------------

In [12]:
"""
Publication-quality figures for significant multinomial logit predictors.
Reference class: Content_Sharing

Outputs:
  - forest_plot.png      : grouped forest plot (features as groups, discourse type as colour)
  - dot_matrix.png       : dot matrix (features × discourse types)
  - combined_plots.png   : both side-by-side on one figure

Run:  python plot_significant.py
Dependencies: matplotlib, numpy
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

# ── Data ──────────────────────────────────────────────────────────────────────
# (discourse_type, feature, coef, se, sig)
DATA = [
    ("Information Seeking",  "Higher Tenant Satisfaction", -0.920, 0.460, "*"  ),
    ("Information Seeking",  "Larger HA's",        0.980, 0.400, "*"  ),
    ("Off-topic Complaints", "Higher Tenant Satisfaction", -1.200, 0.580, "*"  ),
    ("Off-topic Complaints", "Medium-High Affordability",       -0.950, 0.460, "*"  ),
    ("On-topic Criticism",   "Larger HA's",        1.200, 0.390, "**" ),
    ("On-topic Criticism",   "Medium-High Affordability",       -1.050, 0.380, "**" ),
    ("On-topic Feedbacks",   "Medium-High Affordability",       -0.850, 0.410, "*"  ),
    ("On-topic Praise",      "Higher Tenant Satisfaction", -1.300, 0.590, "*"  ),
]

FEATURES_ORDER = [
    "Larger HA's",
    "Medium-High Affordability",
    "Higher Tenant Satisfaction",
]
DTS_ORDER = [
    "Information Seeking",
    "Off-topic Complaints",
    "On-topic Criticism",
    "On-topic Feedbacks",
    "On-topic Praise",
]

PALETTE = {
    "Information Seeking":  "#185FA5",
    "Off-topic Complaints": "#D85A30",
    "On-topic Criticism":   "#993356",
    "On-topic Feedbacks":   "#1D9E75",
    "On-topic Praise":      "#BA7517",
}

BG      = "#FAFAF8"
GRID_C  = "#E0DED6"
ZERO_C  = "#888780"
TEXT_D  = "#2C2C2A"
TEXT_M  = "#5F5E5A"
TEXT_L  = "#888780"

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 1 — FOREST PLOT
# ─────────────────────────────────────────────────────────────────────────────
def make_forest(ax):
    feat_clean = [f.replace("\n", " ") for f in FEATURES_ORDER]
    n_feats = len(FEATURES_ORDER)
    n_dts   = len(DTS_ORDER)

    # vertical slot per feature group; discourse types spread within group
    group_height = 1.2
    dt_step      = group_height / (n_dts + 1)

    y_ticks, y_labels = [], []

    for fi, feat in enumerate(FEATURES_ORDER):
        group_center = fi * (n_dts * dt_step + 0.9)

        # shaded band per feature group
        band_lo = group_center - 0.45
        band_hi = group_center + (n_dts - 1) * dt_step + 0.45
        ax.axhspan(band_lo, band_hi,
                   color="#F1EFE8" if fi % 2 == 0 else BG,
                   zorder=0, lw=0)

        # feature group header
        ax.text(-2.75, group_center + (n_dts - 1) * dt_step / 2,
                feat_clean[fi],
                va="center", ha="left", fontsize=9.5, fontweight="bold",
                color=TEXT_D,
                bbox=dict(boxstyle="round,pad=0.28", facecolor="#E8E6DE",
                          edgecolor="none"))

        rows = [d for d in DATA if d[1] == feat]
        dt_in_feat = [d[0] for d in rows]

        for di, dt in enumerate(DTS_ORDER):
            row_match = [d for d in rows if d[0] == dt]
            y = group_center + di * dt_step
            color = PALETTE[dt]

            if row_match:
                _, _, coef, se, sig = row_match[0]
                ci_lo = coef - 1.96 * se
                ci_hi = coef + 1.96 * se

                ax.plot([ci_lo, ci_hi], [y, y],
                        color=color, linewidth=2.0, zorder=3,
                        solid_capstyle="round")
                for xc in [ci_lo, ci_hi]:
                    ax.plot([xc, xc], [y - 0.10, y + 0.10],
                            color=color, linewidth=1.6, zorder=3)
                ax.scatter(coef, y, color=color, s=72,
                           edgecolors="white", linewidths=0.9, zorder=4)
                ax.text(ci_hi + 0.07, y, sig,
                        va="center", ha="left", fontsize=9.5,
                        color=color, fontweight="bold")
                ax.text(-2.75, y, f"{coef:+.3f}",
                        va="center", ha="left", fontsize=8,
                        color=color, alpha=0.80,
                        fontfamily="monospace")
            else:
                ax.scatter(0, y, color=GRID_C, s=18, zorder=2, marker="o")

            y_ticks.append(y)
            y_labels.append(dt)

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=9)
    for tick, dt_name in zip(ax.get_yticklabels(), y_labels):
        tick.set_color(PALETTE.get(dt_name, TEXT_M))

    ax.axvline(0, color=ZERO_C, linewidth=0.9, linestyle="--",
               zorder=1, alpha=0.8)
    ax.set_xlim(-3.1, 2.8)
    ax.set_xlabel("Log-odds coefficient  ±  95% CI", fontsize=10,
                  color=TEXT_M, labelpad=6)
    ax.xaxis.set_tick_params(labelsize=8.5, colors=TEXT_L)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color(GRID_C)
    ax.grid(axis="x", color=GRID_C, linestyle=":", linewidth=0.7,
            alpha=0.7, zorder=0)

    total_h = (n_feats - 1) * (n_dts * dt_step + 0.9) + (n_dts - 1) * dt_step
    ax.set_ylim(-0.7, total_h + 0.7)
    ax.set_facecolor(BG)

    legend_handles = [
        mpatches.Patch(facecolor=PALETTE[dt], edgecolor="none", label=dt)
        for dt in DTS_ORDER
    ]
    legend_handles += [
        Line2D([0],[0], linestyle="none", marker="none", label=""),
        Line2D([0],[0], linestyle="none", marker="none",
               label="* p<0.05   ** p<0.01", color=TEXT_L),
    ]
    leg = ax.legend(handles=legend_handles, title="Discourse type",
                    title_fontsize=8.5, fontsize=8, loc="lower right",
                    frameon=True, framealpha=0.93,
                    edgecolor=GRID_C, facecolor=BG,
                    handlelength=0.9, handleheight=0.85)
    leg.get_title().set_color(TEXT_M)
    for t in leg.get_texts():
        t.set_color(TEXT_M)

    ax.set_title("Forest plot — significant predictors\n"
                 "Reference: Content Sharing",
                 fontsize=11, fontweight="bold", color=TEXT_D,
                 pad=10, loc="left")


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 2 — DOT MATRIX
# ─────────────────────────────────────────────────────────────────────────────
def make_dot_matrix(ax):
    n_feats = len(FEATURES_ORDER)
    n_dts   = len(DTS_ORDER)

    # grid lines
    for i in range(n_feats + 1):
        ax.axvline(i - 0.5, color=GRID_C, linewidth=0.5, zorder=0)
    for j in range(n_dts + 1):
        ax.axhline(j - 0.5, color=GRID_C, linewidth=0.5, zorder=0)

    # alternating column bands
    for fi in range(n_feats):
        if fi % 2 == 0:
            ax.axvspan(fi - 0.5, fi + 0.5, color="#F1EFE8", zorder=0, lw=0)

    lookup = {(d[0], d[1]): d for d in DATA}

    for fi, feat in enumerate(FEATURES_ORDER):
        for di, dt in enumerate(DTS_ORDER):
            y = n_dts - 1 - di       # top-to-bottom
            x = fi

            key = (dt, feat)
            if key in lookup:
                _, _, coef, se, sig = lookup[key]
                ispos  = coef > 0
                color  = "#185FA5" if ispos else "#D85A30"
                size   = 420 if abs(coef) >= 1.1 else (260 if abs(coef) >= 0.85 else 140)

                ax.scatter(x, y, s=size, color=color,
                           edgecolors="white", linewidths=0.9, zorder=3)

                ax.text(x, y + 0.22, sig,
                        ha="center", va="bottom",
                        fontsize=10,                 # larger stars
                        fontweight="normal",
                        color="black",               # black text
                        zorder=4)

                ax.text(x, y - 0.26, f"{coef:+.2f}",
                        ha="center", va="top",
                        fontsize=10,                 # larger coefficient numbers
                        fontweight="normal",           # optional
                        color="black",               # black numbers
                        fontfamily="monospace",
                        zorder=4)
            else:
                ax.scatter(x, y, s=40, color=GRID_C,
                           marker="o", zorder=2, alpha=0.6)

    # x-axis: wrap long labels onto two lines to avoid overlap
    feat_wrapped = [
    "Larger HA's",
    "Medium-High\nAffordability",
    "Higher Tenant\nSatisfaction"
]
    ax.set_xticks(range(n_feats))
    ax.set_xticklabels(
        feat_wrapped,
        fontsize=10,          # larger x-axis labels
        color="black",        # black x-axis text
        fontweight="normal",
        ha="center",
        multialignment="center"
    )
    ax.xaxis.set_tick_params(length=0)
    ax.tick_params(axis="x", pad=10)

    ax.set_yticks(range(n_dts))
    ax.set_yticklabels(
        list(reversed(DTS_ORDER)),
        fontsize=10,          # larger y-axis labels
        color="black",        # black y-axis text
        fontweight="normal"
    )
    for tick in ax.get_yticklabels():
        tick.set_color("black")
    ax.yaxis.set_tick_params(length=0)

    ax.set_xlim(-0.55, n_feats - 0.45)
    ax.set_ylim(-0.7, n_dts - 0.3)
    ax.spines[["top", "right", "bottom", "left"]].set_visible(False)
    ax.set_facecolor(BG)

    ax.set_title("Dot matrix — coefficient direction & magnitude\nReference: Content Sharing",
                 fontsize=11, fontweight="bold", color=TEXT_D, pad=14, loc="left")

    # clean 3-item legend: direction + significance note only
    legend_handles = [
        mpatches.Patch(facecolor="#185FA5", edgecolor="none", label="Positive effect"),
        mpatches.Patch(facecolor="#D85A30", edgecolor="none", label="Negative effect"),
        Line2D([0],[0], linestyle="none", marker="none",
               label="Dot size = |coefficient|", color=TEXT_L),
        Line2D([0],[0], linestyle="none", marker="none",
               label="* p<0.05   ** p<0.01", color=TEXT_L),
    ]
    leg = ax.legend(handles=legend_handles, ncol=4,
                    loc="upper center", bbox_to_anchor=(0.5, -0.22),
                    fontsize=9, frameon=True, framealpha=0.93,
                    edgecolor=GRID_C, facecolor=BG,
                    handlelength=1.0, handleheight=1.0,
                    columnspacing=1.4)
    for t in leg.get_texts():
        t.set_color(TEXT_M)


# ─────────────────────────────────────────────────────────────────────────────
# RENDER & SAVE
# ─────────────────────────────────────────────────────────────────────────────
def render_forest_standalone():
    fig, ax = plt.subplots(figsize=(10, 7))
    fig.patch.set_facecolor(BG)
    make_forest(ax)
    plt.tight_layout()
    fig.savefig("forest_plot.png", dpi=300,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print("Saved → forest_plot.png")


def render_matrix_standalone():
    fig, ax = plt.subplots(figsize=(8, 6.5))
    fig.patch.set_facecolor(BG)
    make_dot_matrix(ax)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.30)
    fig.savefig("dot_matrix.png", dpi=300,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print("Saved → dot_matrix.png")


def render_combined():
    fig = plt.figure(figsize=(18, 8))
    fig.patch.set_facecolor(BG)
    gs = gridspec.GridSpec(1, 2, width_ratios=[1.15, 1], wspace=0.10)

    ax_forest = fig.add_subplot(gs[0])
    ax_matrix = fig.add_subplot(gs[1])

    make_forest(ax_forest)
    make_dot_matrix(ax_matrix)

    fig.suptitle(
        "Significant predictors of discourse type  |  Multinomial logistic regression\n"
        "Reference class: Content Sharing   |   Only p < 0.05 shown",
        fontsize=12, fontweight="bold", color=TEXT_D, y=1.01
    )

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    fig.savefig("combined_plots.png", dpi=300,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print("Saved → combined_plots.png")


if __name__ == "__main__":
    render_forest_standalone()
    render_matrix_standalone()
    render_combined()
    print("Done!")

Saved → forest_plot.png
Saved → dot_matrix.png


C:\Users\20245179\AppData\Local\Temp\ipykernel_19840\1998751852.py:325: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved → combined_plots.png
Done!
